# 🏠 End-to-End House Price Predictor
## ⚙️ Phase 3: Feature Engineering

**Author**: Lakshit  
**Stage**: Phase 3 of 7  
**Goal**: Transform cleaned raw data into a high-signal, model-ready feature matrix for linear models and tree-based ensembles.

---

### Why Feature Engineering Matters
> *"Coming up with features is difficult, time-consuming, requires expert knowledge. 'Applied machine learning' is basically feature engineering."* — Prof. Andrew Ng

In this phase, we move beyond raw columns to construct features that capture **real estate domain logic**:
1. **Total Living Space (`TotalSF`)**: Appraisers value total usable area (Basement + 1st Floor + 2nd Floor).
2. **Effective Bathrooms (`TotalBath`)**: Half-baths add value, but roughly half as much as full baths.
3. **House & Remodel Age**: Age at time of sale is vastly more informative than raw calendar years.
4. **Ordinal Quality Encodings**: Preserving natural order (`Ex > Gd > TA > Fa > Po`) without creating dozens of sparse dummy columns.
5. **Log-Transforming Skewed Numerical Predictors**: Mitigating extreme leverage points for linear models.
6. **One-Hot Encoding & Train-Test Column Alignment**: Ensuring categorical parity between training data and test submissions.


---
### 1. Setup & Imports
We configure visualization libraries, set reproducible display settings, and import both pandas/numpy and our custom `src/` modules.


In [ ]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Add project root to sys.path for src/ imports
ROOT_DIR = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(ROOT_DIR) not in sys.path:
    sys.path.insert(0, str(ROOT_DIR))

import src.features as feat

# Plotting aesthetics
plt.style.use("seaborn-v0_8-whitegrid" if "seaborn-v0_8-whitegrid" in plt.style.available else "default")
plt.rcParams["figure.figsize"] = (10, 5)
plt.rcParams["font.size"] = 10

DATA_DIR = ROOT_DIR / "data" / "processed"
print(f"Data directory: {DATA_DIR}")


---
### 2. Load Cleaned Data from Phase 2
We load `train_cleaned.csv` and `test_cleaned.csv` produced during Phase 2 Data Cleaning.
Notice we set `keep_default_na=False` so that strings like `'None'` (meaning 'No Pool' or 'No Garage') remain valid string categories rather than converting to NaN.


In [ ]:
train_cleaned = pd.read_csv(DATA_DIR / "train_cleaned.csv", keep_default_na=False)
test_cleaned  = pd.read_csv(DATA_DIR / "test_cleaned.csv", keep_default_na=False)

print(f"Loaded Train: {train_cleaned.shape[0]:,} rows × {train_cleaned.shape[1]} columns")
print(f"Loaded Test : {test_cleaned.shape[0]:,} rows × {test_cleaned.shape[1]} columns")
print(f"Train missing values: {train_cleaned.isnull().sum().sum()}")
print(f"Test missing values : {test_cleaned.isnull().sum().sum()}")


---
### 3. Step 1: Real Estate Domain Feature Creation
In real estate appraisal, square footage and age are evaluated holistically rather than in isolation:

1. **`TotalSF`**: Aggregates `TotalBsmtSF` + `1stFlrSF` + `2ndFlrSF`. In Phase 1 EDA, living area had a 0.70+ correlation with price. Total combined floor area captures the true usable size of the house.
2. **`TotalBath`**: Full baths + $0.5 \times$ half baths + basement full baths + $0.5 \times$ basement half baths.
3. **`HouseAge` & `RemodAge`**: `YrSold - YearBuilt` and `YrSold - YearRemodAdd`. Raw years (e.g. 1995 vs 2005) are tricky for models to extrapolate, but age at sale directly captures physical depreciation.
4. **`IsRemodeled`**: Binary flag indicating if the house was remodeled after its original build year.
5. **Presence Flags**: `HasPool`, `HasGarage`, `Has2ndFloor`, `HasFireplace`, `HasBsmt`.


In [ ]:
train_domain = feat.create_domain_features(train_cleaned)
test_domain  = feat.create_domain_features(test_cleaned)

# Inspect correlations of our newly engineered features with log(SalePrice)
domain_cols = [
    "TotalSF", "TotalBath", "HouseAge", "RemodAge", "IsRemodeled",
    "HasPool", "HasGarage", "Has2ndFloor", "HasFireplace", "HasBsmt",
    "OverallScore", "TotalPorchSF"
]

corrs = train_domain[domain_cols + ["SalePrice"]].corr()["SalePrice"].sort_values(ascending=False)
print("Correlation of New Domain Features with log(SalePrice):")
print("-" * 55)
for col, r in corrs.items():
    if col != "SalePrice":
        print(f"  {col:<16} : {r:+.4f}")


Notice how strong these engineered domain features are:
- `TotalSF` achieves **r = +0.818**, outperforming individual square footage columns!
- `TotalBath` achieves **r = +0.677**.
- `HouseAge` and `RemodAge` exhibit strong negative correlations (**r = -0.588** and **-0.569**), confirming that older houses systematically sell for less unless remodeled.

Let's visualize the relationship between `TotalSF` and `SalePrice`.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.scatterplot(
    data=train_domain, x="TotalSF", y="SalePrice",
    hue="OverallQual", palette="viridis", alpha=0.7, ax=axes[0]
)
axes[0].set_title("TotalSF vs log(SalePrice) (Color = OverallQual)", fontweight="bold")
axes[0].set_xlabel("Total Usable Square Footage (Basement + 1st + 2nd)")
axes[0].set_ylabel("log(SalePrice)")

sns.scatterplot(
    data=train_domain, x="HouseAge", y="SalePrice",
    alpha=0.6, color="#e74c3c", ax=axes[1]
)
axes[1].set_title("HouseAge at Sale vs log(SalePrice)", fontweight="bold")
axes[1].set_xlabel("House Age (Years)")
axes[1].set_ylabel("log(SalePrice)")

plt.tight_layout()
plt.show()


---
### 4. Step 2: Ordinal Feature Encoding
#### Nominal vs. Ordinal Categoricals
- **Nominal** (e.g. `Neighborhood`, `MSZoning`): No natural ordering. A house in `CollgCr` is not intrinsically "greater than" `Veenker`. Must be One-Hot encoded.
- **Ordinal** (e.g. `ExterQual`, `KitchenQual`, `BsmtExposure`): Clear, strict hierarchical progression (`Ex > Gd > TA > Fa > Po > None`).

#### Why Not One-Hot Encode Ordinals?
1. **Curse of Dimensionality**: One-hot encoding 15 ordinal columns creates ~70 sparse binary columns.
2. **Loss of Ordered Information**: An integer encoding ($0, 1, 2, 3, 4, 5$) informs linear and tree models that $Ex$ is adjacent to $Gd$ and superior to $Fa$.


In [ ]:
train_ord, test_ord = feat.encode_ordinal_features(train_domain, test_domain)

ord_cols_sample = ["ExterQual", "KitchenQual", "BsmtQual", "FireplaceQu", "HeatingQC", "GarageFinish"]
print("Sample encoded ordinal distributions in Train:")
print(train_ord[ord_cols_sample].describe().round(2).T[["mean", "std", "min", "50%", "max"]])


---
### 5. Step 3: Handling Skewness in Continuous Features
#### Why Feature Skewness Matters
Linear regression, Ridge, and Lasso assume features have relatively symmetric distributions with consistent variance (homoscedasticity). Features with severe positive skew (like `LotArea` where a few mansions have 100,000+ sq ft) exert **massive leverage** on model coefficients, pulling regression lines towards outliers.

Applying `np.log1p(x) = log(1 + x)`:
1. Compresses long right tails into near-normal distributions.
2. Handles zeros gracefully (e.g., houses with 0 sq ft basement or 0 wood deck).
3. We use the established threshold: $|\text{skewness}| > 0.75$.


In [ ]:
train_skewed, test_skewed, transformed_cols = feat.log_transform_skewed(train_ord, test_ord, threshold=0.75)

print(f"Features transformed: {len(transformed_cols)}")
print("First 10 transformed features:", transformed_cols[:10])

# Compare before vs after skewness for 4 key features
sample_skew_cols = ["LotArea", "1stFlrSF", "GrLivArea", "TotalSF"]
print(f"\n{'Feature':<15} | {'Raw Skew':<10} | {'Log1p Skew':<10}")
print("-" * 42)
for col in sample_skew_cols:
    raw_s = train_ord[col].skew()
    log_s = train_skewed[col].skew()
    print(f"{col:<15} | {raw_s:10.3f} | {log_s:10.3f}")


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 8))

sns.histplot(train_ord["LotArea"], kde=True, ax=axes[0, 0], color="#3498db")
axes[0, 0].set_title(f"LotArea (Raw) — Skew: {train_ord['LotArea'].skew():.2f}", fontweight="bold")

sns.histplot(train_skewed["LotArea"], kde=True, ax=axes[0, 1], color="#2ecc71")
axes[0, 1].set_title(f"LotArea (log1p) — Skew: {train_skewed['LotArea'].skew():.2f}", fontweight="bold")

sns.histplot(train_ord["TotalSF"], kde=True, ax=axes[1, 0], color="#e67e22")
axes[1, 0].set_title(f"TotalSF (Raw) — Skew: {train_ord['TotalSF'].skew():.2f}", fontweight="bold")

sns.histplot(train_skewed["TotalSF"], kde=True, ax=axes[1, 1], color="#9b59b6")
axes[1, 1].set_title(f"TotalSF (log1p) — Skew: {train_skewed['TotalSF'].skew():.2f}", fontweight="bold")

plt.tight_layout()
plt.show()


---
### 6. Step 4: One-Hot Encoding & Column Alignment
#### The Dummy Variable Alignment Trap
When one-hot encoding categories with `pd.get_dummies()`:
- `test.csv` might not contain every rare category present in `train.csv`.
- If encoded separately, `train` and `test` will end up with different columns and shapes, causing models to crash at prediction time!

#### The Solution: Strict Column Alignment
```python
X_tr_aligned, X_te_aligned = X_tr_enc.align(X_te_enc, join='left', axis=1, fill_value=0)
```
- `join='left'`: `test` retains exactly the columns present in `train`.
- `fill_value=0`: Any category column missing in test is filled with 0 (absence of category).
- Any novel category in test is safely discarded since the model never trained on it.
- `drop_first=True`: Avoids the dummy variable trap (multicollinearity).


In [ ]:
train_final, test_final = feat.one_hot_encode_and_align(train_skewed, test_skewed)

print(f"Final Train Shape: {train_final.shape}")
print(f"Final Test Shape : {test_final.shape}")
print(f"Missing values in Train: {train_final.isnull().sum().sum()}")
print(f"Missing values in Test : {test_final.isnull().sum().sum()}")
assert train_final.drop(columns=['SalePrice']).columns.equals(test_final.columns), "Columns do not match!"
print("✓ Column alignment check PASSED: Train features and Test features are identical!")


---
### 7. Step 5: Feature Correlation with Target
Let's inspect the top 15 features most strongly correlated with our target `SalePrice` (log-scale).


In [ ]:
model_feats = [c for c in train_final.columns if c not in ["Id", "SalePrice"]]
correlations = train_final[model_feats].corrwith(train_final["SalePrice"]).sort_values()

top_neg = correlations.head(7)
top_pos = correlations.tail(10)
top_combined = pd.concat([top_neg, top_pos])

plt.figure(figsize=(10, 7))
colors = ["#e74c3c" if val < 0 else "#27ae60" for val in top_combined.values]
top_combined.plot(kind="barh", color=colors)
plt.axvline(0, color="black", linestyle="--", alpha=0.7)
plt.title("Top Feature Correlations with log(SalePrice)", fontsize=13, fontweight="bold")
plt.xlabel("Pearson Correlation (r)")
plt.tight_layout()
plt.show()


---
### 8. Step 6: Verify Reusable `src/features.py` Pipeline
We ensure the entire transformation can be executed cleanly from start to finish via `src/features.py`. This guarantees seamless automation in our upcoming modeling and deployment phases.


In [ ]:
# Run full pipeline from raw cleaned datasets
df_tr = feat.create_domain_features(train_cleaned)
df_te = feat.create_domain_features(test_cleaned)
df_tr, df_te = feat.encode_ordinal_features(df_tr, df_te)
df_tr, df_te, _ = feat.log_transform_skewed(df_tr, df_te)
df_tr_final, df_te_final = feat.one_hot_encode_and_align(df_tr, df_te)

# Display executive report
feat.feature_engineering_report(df_tr_final, df_te_final)

# Save to data/processed/
df_tr_final.to_csv(DATA_DIR / "train_features.csv", index=False)
df_te_final.to_csv(DATA_DIR / "test_features.csv", index=False)
print("✓ Successfully saved train_features.csv and test_features.csv to data/processed/")


---
## 🎯 Phase 3 Summary & Next Steps

### What We Accomplished
1. **Domain Feature Engineering**: Built 12 high-impact real estate domain features (`TotalSF`, `TotalBath`, `HouseAge`, `RemodAge`, `OverallScore`, binary flags).
2. **Ordinal Mapping**: Encoded 19 ordered ratings to integer scales, saving 70+ sparse one-hot columns while preserving monotonic hierarchy.
3. **Log1p Skewness Normalization**: Normalized 20 continuous right-skewed features ($|\text{skew}| > 0.75$), mitigating leverage points for regularized linear models.
4. **Categorical One-Hot Encoding**: Dummified nominal variables with `drop_first=True`.
5. **Train-Test Parity**: Aligned 192 model features identically between train and test with zero null values.
6. **Production Module**: Encapsulated all transformations into modular, reusable functions in `src/features.py`.

---

### Ready for Phase 4: Baseline Modeling
In **Phase 4**, we will:
- Partition `train_features.csv` into 80/20 train/validation splits.
- Train four baseline models: `LinearRegression`, `Ridge`, `Lasso`, and `DecisionTreeRegressor`.
- Perform 5-fold cross-validation and evaluate RMSE and $R^2$.
- Benchmark regularization effects and save the top-performing baseline model.
